In [ ]:
# run this notebook after 3_call_ibd2_using_snipar_python
# use this notebook to combine ibd calls across chromosomes
# after this, run 5_subset_vds_python 

In [ ]:
library(data.table)
library(dplyr)
install.packages('R.utils')

In [ ]:
chr = 1
ibd_segs = fread(paste("./snipar_out_dec20/sib_snipar_chr_", chr, ".ibd.segments.gz", sep = ""))

for (i in 1:22){
    chr = i 
    sub = fread(paste("./snipar_out_dec20/sib_snipar_chr_", chr, ".ibd.segments.gz", sep = ""))
    ibd_segs = rbind(ibd_segs, sub)
}

In [ ]:
ibd_segs_rev = ibd_segs 
ibd_segs_rev$ID1 = ibd_segs$ID2
ibd_segs_rev$ID2 = ibd_segs$ID1
ibd_segs = rbind(ibd_segs_rev, ibd_segs)
ibd_segs = ibd_segs %>% filter(ID1 < ID2)
ibd_segs$length = ibd_segs$stop_coordinate-ibd_segs$start_coordinate + 1
fwrite(ibd_segs, "ibd_all_dec20.txt", row.names = F, quote = F, sep = "\t")
ibd_segs2 = ibd_segs %>% filter(IBDType == 2)
fwrite(ibd_segs2, "ibd2_all_dec20.txt", row.names = F, quote = F, sep = "\t")

In [ ]:
# don't forget to wget https://hgdownload.soe.ucsc.edu/goldenpath/hg38/bigZips/hg38.chrom.sizes
# read in chr sizes 
chr_sizes = fread("./hg38.chrom.sizes")
chr_sizes$chr = as.numeric(gsub(x = chr_sizes$V1, pattern = "chr", replace = ""))
chr_sizes = chr_sizes %>% filter(chr %in% 1:22)

In [ ]:
# read in sibs 
sibs = fread("./relatedness_dec20/sibs.txt")
ibd_stats = data.frame(idv1 = sibs$ID1, idv2 = sibs$ID2, gwide_IBD1 = NA, gwide_IBD2 = NA)
for (i in 1:nrow(sibs)){
    idv1 = sibs$ID1[i]
    idv2 = sibs$ID2[i]
    sub_sib = ibd_segs %>% filter(ID1 == idv1 & ID2 == idv2)
    ibd1 = rep(0, 22)
    ibd2 = rep(0, 22)
    for (j in 1:22){
        chr_len = chr_sizes$V2[which(chr_sizes$chr == j)]
        sub_sib_chr = sub_sib %>% filter(Chr == j)
        len_ibd1 = sum(sub_sib_chr$length[which(sub_sib_chr$IBDType == 1)])
        len_ibd2 = sum(sub_sib_chr$length[which(sub_sib_chr$IBDType == 2)])
        ibd1[j] = len_ibd1/chr_len
        ibd2[j] = len_ibd2/chr_len
    }
    ibd_stats$gwide_IBD1[i] = mean(ibd1)
    ibd_stats$gwide_IBD2[i] = mean(ibd2)
}
fwrite(ibd_stats, "./ibd_stats.txt", sep = "\t", row.names = F, quote = F)